# b2 · Elastic Net forecasts

One-month-ahead log realized variance for the five cyclical ETFs. Four Elastic Net variants share one walk-forward engine using an expanding window, monthly refit, and 60-month initial training burn-in window.

| Variant | Features |
|---|---|
| `en_full_macro` | 11 macro variables + `log_rv_1m` |
| `en_no_vix` | full macro without `VIXCLS` |
| `en_vix_only` | `VIXCLS` + `log_rv_1m` |
| `en_no_macro` | `log_rv_1m` only |

Output: `forecasts/en_forecasts.csv`. Tests are in `evaluation/e3_vix_effect_en.ipynb`.

In [7]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, enet_path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


MACRO_PATH = os.path.join(PROJECT_ROOT, "dataset", "macro_variables", "final_clean", "final_macro_data.csv")
RV_PATH = os.path.join(PROJECT_ROOT, "dataset", "price", "clean", "realized_variance.csv")
FORECASTS_DIR = os.path.join(PROJECT_ROOT, "models", "forecasting", "forecasts")

CYCLICAL_TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]
BURN_IN = 60        # months of history before the first forecast
REFIT_EVERY = 1     # refit and re-tune every month
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

## Data

The 11 macro variables from the proposal, plus `log_rv_1m` from the shared `realized_variance.csv`. Row `t` holds information up to month `t`; `y_target` is `log_rv_1m` of month `t+1`.

In [8]:
FINAL_MACRO_VARS = [
    "T10Y2Y", "DGS3MO", "BAA10Y", "CPIAUCSL", "T5YIE", "T5YIE_MINUS_T10YIE",
    "VIXCLS", "INDPRO", "UNRATE", "UMCSENT", "DCOILWTICO"
]

macro = pd.read_csv(MACRO_PATH, index_col="date", parse_dates=True)[FINAL_MACRO_VARS]
rv = pd.read_csv(RV_PATH, parse_dates=["date"])

# the last month has no y_target, each model drops it with dropna() below
aligned_data = {}
for ticker in CYCLICAL_TICKERS:
    rv_t = rv.loc[rv["etf"] == ticker, ["date", "log_rv_1m"]].set_index("date").sort_index()
    y_target = rv_t["log_rv_1m"].shift(-1).rename("y_target")
    aligned_data[ticker] = macro.join(rv_t, how="inner").join(y_target, how="inner")
    idx = aligned_data[ticker].index
    print(f"{ticker}: {len(idx)} rows, {idx.min().date()} to {idx.max().date()}")

XLI: 275 rows, 2003-02-28 to 2025-12-31
XLB: 275 rows, 2003-02-28 to 2025-12-31
XLE: 275 rows, 2003-02-28 to 2025-12-31
XLY: 275 rows, 2003-02-28 to 2025-12-31
XLF: 275 rows, 2003-02-28 to 2025-12-31


## Variants

In [9]:
NO_VIX_MACRO = [c for c in FINAL_MACRO_VARS if c != "VIXCLS"]

FEATURE_SETS = {
    "en_full_macro": FINAL_MACRO_VARS + ["log_rv_1m"],
    "en_no_vix": NO_VIX_MACRO + ["log_rv_1m"],
    "en_vix_only": ["VIXCLS", "log_rv_1m"],
    "en_no_macro": ["log_rv_1m"],
}
for name, cols in FEATURE_SETS.items():
    print(f"{name:18s} {len(cols):2d} features")

en_full_macro      12 features
en_no_vix          11 features
en_vix_only         2 features
en_no_macro         1 features


## Recursive forecasts

Each model is estimated on an expanding window and refit every month, starting after a 60-month burn-in. In month *t* the model uses only data up to *t*. `alpha` and `l1_ratio` are chosen by 5-fold `TimeSeriesSplit`, with the scaler fitted on each fold's training part only. The selected model, scaler included, is then refit on the full window and forecasts log RV for month *t+1*.

- `sigma2`: variance of the out-of-fold residuals from the same CV splits.
- `fc_level = exp(fc_log + sigma2 / 2)`: log-normal bias correction, used for QLIKE.

In [10]:
def tune_enet(X_train, y_train, l1_ratios, n_splits=5, n_alphas=100, eps=1e-3):
    # Pick (alpha, l1_ratio) by time-series CV. The scaler is fitted on each fold's
    # training part only, so validation months never influence their own scaling.
    folds = list(TimeSeriesSplit(n_splits=n_splits).split(X_train))
    # alpha grid as in ElasticNetCV: alpha_max down to eps * alpha_max, for each l1_ratio
    X_std = StandardScaler().fit_transform(X_train)
    alpha_max = np.abs(X_std.T @ (y_train - y_train.mean())).max() / len(y_train)
    best = None
    for l1 in l1_ratios:
        alphas = alpha_max / l1 * np.logspace(0, np.log10(eps), n_alphas)
        mse = np.zeros((len(folds), n_alphas))
        for k, (tr, te) in enumerate(folds):
            scaler = StandardScaler().fit(X_train[tr])
            y_mean = y_train[tr].mean()
            _, coefs, _ = enet_path(scaler.transform(X_train[tr]), y_train[tr] - y_mean,
                                    l1_ratio=l1, alphas=alphas, max_iter=10000)
            pred = scaler.transform(X_train[te]) @ coefs + y_mean
            mse[k] = ((y_train[te][:, None] - pred) ** 2).mean(axis=0)
        cv_mse = mse.mean(axis=0)
        j = int(cv_mse.argmin())
        if best is None or cv_mse[j] < best[0]:
            best = (cv_mse[j], alphas[j], l1)
    return best[1], best[2]


def walk_forward(model_df, feature_cols, burn_in=BURN_IN, refit_every=REFIT_EVERY, l1_ratios=L1_RATIOS,
                 window=None):
    # expanding-window forecasts for one model and ticker (window=None);
    # with window=n, each fit uses only the latest n months instead
    # model_df has no NaNs in feature_cols or y_target.
    X = model_df[feature_cols].to_numpy()
    y = model_df["y_target"].to_numpy()
    dates = model_df.index
    rows = []
    pipe, sigma2 = None, np.nan
    alpha_ = l1_ratio_ = np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        for i in range(burn_in, len(model_df)):
            refit = (i - burn_in) % refit_every == 0
            if refit:
                start = 0 if window is None else max(0, i - window)
                X_train, y_train = X[start:i], y[start:i]
                alpha_, l1_ratio_ = tune_enet(X_train, y_train, l1_ratios)
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", ElasticNet(alpha=alpha_, l1_ratio=l1_ratio_, max_iter=10000, random_state=0)),
                ])

                # Out-of-fold residuals, with the scaler refit inside each fold
                # TimeSeriesSplit never tests the first chunk so skip those rows
                oof = np.full(len(y_train), np.nan)
                for tr, te in TimeSeriesSplit(n_splits=5).split(X_train):
                    oof[te] = clone(pipe).fit(X_train[tr], y_train[tr]).predict(X_train[te])
                tested = ~np.isnan(oof)
                sigma2 = float(np.var(y_train[tested] - oof[tested]))
                pipe.fit(X_train, y_train)

            pred_log = pipe.predict(X[i:i + 1])[0]
            rows.append({
                "date": dates[i],
                "target_date": dates[i] + pd.offsets.MonthEnd(1),
                "actual_logRV": y[i],
                "actual_RV": np.exp(y[i]),
                "fc_log": pred_log,
                "sigma2": sigma2,
                "fc_level": np.exp(pred_log + 0.5 * sigma2),
                "alpha": alpha_,
                "l1_ratio": l1_ratio_,
            })
    return pd.DataFrame(rows).set_index("date")


## Run

A full run is slow because every model is re-tuned every month. With `RECOMPUTE_FORECASTS = False` the saved file is loaded instead. The pipeline runner sets `DSE4101_RECOMPUTE_FORECASTS=1` to force a rebuild.

In [ ]:
TIDY_COLS = ["date", "target_date", "actual_logRV", "actual_RV", "fc_log", "sigma2", "fc_level"]
SAVED_PATH = os.path.join(FORECASTS_DIR, "en_forecasts.csv")


def run_all_models():
    tidy_rows = []
    t0 = time.time()
    for model_name, feature_cols in FEATURE_SETS.items():
        for ticker in CYCLICAL_TICKERS:
            model_df = aligned_data[ticker][feature_cols + ["y_target"]].dropna()
            tidy = walk_forward(model_df, feature_cols).reset_index()[TIDY_COLS]
            tidy.insert(0, "etf", ticker)
            tidy.insert(0, "model", model_name)
            tidy_rows.append(tidy)
        print(f"{model_name:18s} done, {time.time() - t0:.0f}s")
    tidy_df = pd.concat(tidy_rows, ignore_index=True)
    os.makedirs(FORECASTS_DIR, exist_ok=True)
    tidy_df.to_csv(SAVED_PATH, index=False)
    return tidy_df


RECOMPUTE_FORECASTS = os.environ.get("DSE4101_RECOMPUTE_FORECASTS", "0") == "1"

if RECOMPUTE_FORECASTS or not os.path.exists(SAVED_PATH):
    tidy_df = run_all_models()
else:
    tidy_df = pd.read_csv(SAVED_PATH, parse_dates=["date", "target_date"])
print(f"{len(tidy_df)} rows, models: {tidy_df['model'].nunique()}")

en_full_macro      done, 166s


## Accuracy

Pooled and per-ticker MSE on log RV and QLIKE on the level forecast, for all four variants.

In [ ]:
def qlike(actual_rv, fc_level):
    ratio = actual_rv / fc_level
    return ratio - np.log(ratio) - 1


def mse_qlike_table(tidy_df):
    d = tidy_df.assign(sq_err_log=(tidy_df["actual_logRV"] - tidy_df["fc_log"]) ** 2,
                       qlike=qlike(tidy_df["actual_RV"], tidy_df["fc_level"]))
    agg = dict(MSE_log=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
    per_ticker = d.groupby(["model", "etf"]).agg(**agg).reset_index()
    pooled = d.groupby("model").agg(**agg).reset_index().assign(etf="pooled")
    long = pd.concat([per_ticker, pooled], ignore_index=True)

    order = dict(index=list(FEATURE_SETS), columns=CYCLICAL_TICKERS + ["pooled"])
    mse = long.pivot(index="model", columns="etf", values="MSE_log").reindex(**order)
    ql = long.pivot(index="model", columns="etf", values="QLIKE").reindex(**order)
    return mse, ql


mse_expanding, qlike_expanding = mse_qlike_table(tidy_df)
print("MSE (log RV)")
print(mse_expanding.round(4))
print("\nQLIKE")
print(qlike_expanding.round(4))

MSE (log RV)
etf               XLI     XLB     XLE     XLY     XLF  pooled
model                                                        
en_full_macro  0.4387  0.4198  0.5408  0.4796  0.5772  0.4912
en_no_vix      0.5965  0.4645  0.5505  0.5367  0.6376  0.5571
en_vix_only    0.4593  0.4095  0.4481  0.5103  0.5641  0.4783
en_no_macro    0.5302  0.4621  0.4524  0.5056  0.6655  0.5232

QLIKE
etf               XLI     XLB     XLE     XLY     XLF  pooled
model                                                        
en_full_macro  0.2973  0.2837  0.3932  0.3066  0.4022  0.3366
en_no_vix      0.4407  0.3172  0.4148  0.3424  0.4537  0.3938
en_vix_only    0.2884  0.2549  0.2921  0.2892  0.3517  0.2953
en_no_macro    0.3424  0.2962  0.2966  0.2990  0.4052  0.3279


## Robustness: 60-month rolling window

For the robustness check in `models/forecasting/robustness/`, `en_no_macro` and `en_vix_only` are re-estimated on the latest 60 months only, instead of all history. Everything else is unchanged. The first forecast is still for March 2008, since the burn-in is also 60 months.

Output: `models/forecasting/robustness/forecasts/en_rolling60_forecasts.csv`, with models named `*_roll60`. It is rebuilt when `RECOMPUTE_ROLLING` or `RECOMPUTE_FORECASTS` is set, or when the file is missing. Tests are in `models/forecasting/robustness/r3_rolling_window_en.ipynb`.


In [ ]:
ROLLING_WINDOW = 60
ROLLING_MODELS = ["en_no_macro", "en_vix_only"]
ROLLING_PATH = os.path.join(PROJECT_ROOT, "models", "forecasting", "robustness", "forecasts", "en_rolling60_forecasts.csv")


def run_rolling_models():
    tidy_rows = []
    t0 = time.time()
    for model_name in ROLLING_MODELS:
        feature_cols = FEATURE_SETS[model_name]
        for ticker in CYCLICAL_TICKERS:
            model_df = aligned_data[ticker][feature_cols + ["y_target"]].dropna()
            tidy = walk_forward(model_df, feature_cols, window=ROLLING_WINDOW).reset_index()[TIDY_COLS]
            tidy.insert(0, "etf", ticker)
            tidy.insert(0, "model", f"{model_name}_roll{ROLLING_WINDOW}")
            tidy_rows.append(tidy)
        print(f"{model_name:18s} done, {time.time() - t0:.0f}s")
    rolling_df = pd.concat(tidy_rows, ignore_index=True)
    os.makedirs(os.path.dirname(ROLLING_PATH), exist_ok=True)
    rolling_df.to_csv(ROLLING_PATH, index=False)
    return rolling_df


RECOMPUTE_ROLLING = False

if RECOMPUTE_ROLLING or RECOMPUTE_FORECASTS or not os.path.exists(ROLLING_PATH):
    rolling_df = run_rolling_models()
else:
    rolling_df = pd.read_csv(ROLLING_PATH, parse_dates=["date", "target_date"])
print(f"{len(rolling_df)} rows, models: {sorted(rolling_df['model'].unique())}")

In [ ]:
# Same accuracy table as above, expanding vs rolling, for the two models re-estimated
rolling_named = rolling_df.assign(model=rolling_df["model"].str.replace(f"_roll{ROLLING_WINDOW}", "", regex=False))
mse_rolling, qlike_rolling = mse_qlike_table(rolling_named)

pd.concat({
    "MSE_log expanding": mse_expanding.loc[ROLLING_MODELS],
    "MSE_log rolling": mse_rolling.loc[ROLLING_MODELS],
    "QLIKE expanding": qlike_expanding.loc[ROLLING_MODELS],
    "QLIKE rolling": qlike_rolling.loc[ROLLING_MODELS],
}).round(4)